In [ ]:
# Core libraries
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Sklearn utilities
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    roc_curve,
    precision_recall_curve,
    average_precision_score
)

import warnings
warnings.filterwarnings("ignore")

In [ ]:
# Load dataset
df = pd.read_csv("../data/telco_churn.csv")

# Quick check
df.head()

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
df["Churn"].value_counts(normalize=True)

In [ ]:
df["TotalCharges"].head()

In [ ]:
df["TotalCharges"].dtype

In [ ]:
df[df["TotalCharges"] == " "]

In [ ]:
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")

In [ ]:
df["TotalCharges"].isna().sum()

In [ ]:
df = df.dropna(subset=["TotalCharges"])

In [ ]:
categorical_cols = df.select_dtypes(include=["object"]).columns.tolist()
numerical_cols = df.select_dtypes(include=["int64", "float64"]).columns.tolist()

print("Categorical:", categorical_cols)
print("Numerical:", numerical_cols)

In [ ]:
df = df.drop(columns=["customerID"])

In [ ]:
df["Churn"] = df["Churn"].map({"No": 0, "Yes": 1})
df["Churn"].value_counts()

In [ ]:
X = df.drop("Churn", axis=1)
y = df["Churn"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

print("Train size:", X_train.shape)
print("Test size:", X_test.shape)

print("\nTrain churn distribution:")
print(y_train.value_counts(normalize=True))

print("\nTest churn distribution:")
print(y_test.value_counts(normalize=True))

In [ ]:
categorical_cols = X_train.select_dtypes(include=["object"]).columns.tolist()
numerical_cols = X_train.select_dtypes(include=["int64", "float64"]).columns.tolist()

print("Categorical columns:", categorical_cols)
print("Numerical columns:", numerical_cols)

In [ ]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

In [ ]:
#preprocessing pipeline:
numeric_transformer = StandardScaler()

categorical_transformer = OneHotEncoder(
    handle_unknown="ignore",
    drop=None
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numerical_cols),
        ("cat", categorical_transformer, categorical_cols)
    ]
)

In [ ]:
#pipeline:
model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", LogisticRegression(
            max_iter=1000,
            class_weight="balanced",
            random_state=42
        ))
    ]
)

In [ ]:
#train model:
model.fit(X_train, y_train)

In [ ]:
#predictions:

# Predict probabilities
y_proba = model.predict_proba(X_test)[:, 1]

# Default predictions (threshold = 0.5)
y_pred_default = model.predict(X_test)

In [ ]:
#ROC-AUC
roc_auc = roc_auc_score(y_test, y_proba)
print("ROC-AUC:", roc_auc)

In [ ]:
#PR-AUC
pr_auc = average_precision_score(y_test, y_proba)
print("PR-AUC:", pr_auc)

In [ ]:
#confusion matrix:
print("Confusion Matrix (Default 0.5 threshold):")
print(confusion_matrix(y_test, y_pred_default))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_default))

In [ ]:
import numpy as np
from sklearn.metrics import confusion_matrix

def calculate_profit(y_true, y_proba, threshold,
                     retention_cost=10,
                     revenue=200,
                     retention_success_rate=0.3):
    
    y_pred = (y_proba >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    
    retained_customers = tp * retention_success_rate
    revenue_gain = retained_customers * revenue
    campaign_cost = (tp + fp) * retention_cost
    
    profit = revenue_gain - campaign_cost
    
    return profit, tp, fp, fn, tn


thresholds = np.arange(0.1, 0.9, 0.05)

results = []

for t in thresholds:
    profit, tp, fp, fn, tn = calculate_profit(y_test, y_proba, t)
    results.append((t, profit, tp, fp, fn, tn))

results_df = pd.DataFrame(results, columns=["threshold", "profit", "tp", "fp", "fn", "tn"])

results_df.sort_values("profit", ascending=False).head()